In [ ]:
import os
import glob
import json
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    run_glutamate_tuning_analysis,
)

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))



In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
savepath = r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures'

In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

In [ ]:
st_paths = [glob.glob(os.path.join(asset.derived_dir,'**','glutamate_single_trial_df.npz'),recursive=True)[0] for asset in assets]

In [ ]:
act_summary_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data\activation_summary.csv"
act_summary = pd.read_csv(act_summary_path)

In [ ]:
config = GlutamateAnalysisConfig(
    tuning_method="fve",
    tuning_fve_mode="time_avg",                 # "trace" = no time averaging
    tuning_fve_amplitude_func="mean",        # mainly relevant for time_avg mode
    tuning_fve_sample_slice=(50, 100),       # mainly relevant for time_avg mode
    tuning_response_classes=("activated",),  # note trailing comma
    n_shuffles_tuning=10,                    # keep low while testing
    random_seed=0,
)

In [ ]:
all_tuning_summary = []
all_tuning_per_image = []

for asset in assets:
    print(asset.session_id)
    session_root = asset.session_dir

    results = run_glutamate_tuning_analysis(
        session_dir_or_analysis_dir=session_root,
        activation_summary=act_summary_path,   # or activation_summary_df
        config=config,
        save_tables=False,
    )

    tuning_summary = results["tuning_summary_table"].copy()
    tuning_per_image = results["tuning_per_image_table"].copy()

    dmd_depth_map = {
        "DMD1": asset.metadata.get("dmd1_depth", np.nan),
        "DMD2": asset.metadata.get("dmd2_depth", np.nan),
    }

    if not tuning_summary.empty and "dmd" in tuning_summary.columns:
        tuning_summary["depth"] = tuning_summary["dmd"].map(dmd_depth_map)
        tuning_summary["session_name"] = os.path.basename(session_root)
        all_tuning_summary.append(tuning_summary)
    else:
        print(f"No tuning summary rows for {session_root}")

    if not tuning_per_image.empty and "dmd" in tuning_per_image.columns:
        tuning_per_image["depth"] = tuning_per_image["dmd"].map(dmd_depth_map)
        tuning_per_image["session_name"] = os.path.basename(session_root)
        all_tuning_per_image.append(tuning_per_image)
    else:
        print(f"No tuning per-image rows for {session_root}")

tuning_summary_table = pd.concat(all_tuning_summary, ignore_index=True) if all_tuning_summary else pd.DataFrame()
tuning_per_image_table = pd.concat(all_tuning_per_image, ignore_index=True) if all_tuning_per_image else pd.DataFrame()

In [ ]:
tuning_summary_table.keys()

In [ ]:
plot_df = tuning_summary_table.copy()

In [ ]:
cl, cmap, cp = PNW_cmap.get_PNW_cmap('Sailboat', n_colors=4)
cp

In [ ]:
cp = cp[::-1]

In [ ]:
metric = 'fve_image'

fig,ax=plt.subplots(figsize=(4,4.5))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left','right','top','bottom']:
    ax.spines[spine].set_linewidth(2)

sns.despine()
    
sns.stripplot(data = plot_df,x = 'depth', y = np.log10(plot_df[metric]),palette=cp,size=2)

means = []

for depth in plot_df['depth'].unique():
    dft = plot_df[plot_df['depth']==depth]
    mean = np.nanmean(np.log10(dft[metric]))
    means.append(mean)
ax.plot(means,color='k',zorder=11,lw=3,marker='o')
# ax.set_ylim(-3.75,0.75)

ax.set_ylabel('log(FVE)')
ax.set_xlabel('Depth from pia (\u03BCm)')

ax.set_title('Fraction of response variance\nexplained by the mean')

fig.tight_layout()
filen = 'Image_FVE'
# save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
plot_df.keys()

In [ ]:

fig, ax = plt.subplots(figsize=(4,4))
sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

kde = sns.histplot(
    data=plot_df,
    x=np.log10(plot_df["fve_image"]),
    kde=True,
    ax=ax,
    stat='count',
    bins=150,edgecolor='none',
    common_norm = True
)

for spine in ['left','right','top','bottom']:
    ax.spines[spine].set_linewidth(2)
    
ax.set_xlabel('log(FVE)')
ax.set_ylabel('Synapses')

ax.axvline(np.median(np.log10(plot_df['fve_image'])),color='k',dashes=[3,3])

ax.set_title('Overall variance explained\nby image identity')

fig.tight_layout()

filen = 'overall_image_FVE'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
plot_df.keys()